In [2]:
import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

In [3]:
cap = cv2.VideoCapture("assets/people-walking.mp4")

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Detect only persons
    results = model(frame, classes=[0], verbose=False)

    # Draw detections
    annotated_frame = results[0].plot()

    cv2.imshow("Person Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [4]:
import cv2
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Person-only detection
    results = model(frame, classes=[0], verbose=False)

    for box in results[0].boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        # Calculate centroid
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        # Draw centroid
        cv2.circle(
            frame,
            (cx, cy),
            5,
            (255, 255, 255),
            -1
        )

        # Display coordinates
        cv2.putText(
            frame,
            f"({cx}, {cy})",
            (cx + 10, cy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            2
        )

    cv2.imshow("Centroid Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [9]:
import cv2
from ultralytics import YOLO
import math

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")

# Person ID management
next_id = 1
tracks = {}

MAX_DISTANCE = 80

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, classes=[0], verbose=False)

    current_centroids = []

    # -----------------------------
    # 1. Get current detections
    # -----------------------------
    for box in results[0].boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        current_centroids.append((cx, cy))

        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 2)

    # -----------------------------
    # 2. Match detections to tracks
    # -----------------------------
    updated_tracks = {}

    for centroid in current_centroids:

        best_id = None
        best_distance = MAX_DISTANCE

        for track_id, old_centroid in tracks.items():

            distance = math.sqrt(
                (centroid[0] - old_centroid[0]) ** 2 +
                (centroid[1] - old_centroid[1]) ** 2
            )

            if distance < best_distance:
                best_distance = distance
                best_id = track_id

        # -----------------------------
        # 3. Existing person or new person?
        # -----------------------------
        if best_id is not None:
            track_id = best_id
        else:
            track_id = next_id
            next_id += 1

        updated_tracks[track_id] = centroid

        # Draw ID
        cv2.circle(
            frame,
            centroid,
            5,
            (255, 255, 255),
            -1
        )

        cv2.putText(
            frame,
            f"Person {track_id}",
            (centroid[0] + 10, centroid[1]),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    tracks = updated_tracks

    cv2.imshow("Centroid Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [11]:
import cv2
from ultralytics import YOLO
import math

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")

next_id = 1
tracks = {}

MAX_DISTANCE = 80

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, classes=[0], verbose=False)

    current_centroids = []

    # Get person centroids
    for box in results[0].boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        current_centroids.append((cx, cy))

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

    updated_tracks = {}

    for centroid in current_centroids:

        best_id = None
        best_distance = MAX_DISTANCE

        # Find closest existing track
        for track_id, old_centroid in tracks.items():

            distance = math.dist(
                centroid,
                old_centroid
            )

            if distance < best_distance:
                best_distance = distance
                best_id = track_id

        # --------------------------------
        # Track creation
        # --------------------------------
        if best_id is None:

            track_id = next_id
            next_id += 1

        else:

            track_id = best_id

        # Save track
        updated_tracks[track_id] = centroid

        # Draw ID
        cv2.circle(
            frame,
            centroid,
            5,
            (255, 255, 255),
            -1
        )

        cv2.putText(
            frame,
            f"Person {track_id}",
            (centroid[0] + 10, centroid[1]),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    tracks = updated_tracks

    cv2.imshow("Track Creation", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [14]:
import cv2
from ultralytics import YOLO
import math

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")

next_id = 1
tracks = {}

MAX_DISTANCE = 80

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, classes=[0], verbose=False)

    current_centroids = []

    # Current frame detections
    for box in results[0].boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        current_centroids.append((cx, cy))

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

    updated_tracks = {}

    # Match current detections with previous tracks
    for centroid in current_centroids:

        best_id = None
        best_distance = MAX_DISTANCE

        for track_id, old_centroid in tracks.items():

            distance = math.dist(
                centroid,
                old_centroid
            )

            if distance < best_distance:
                best_distance = distance
                best_id = track_id

        # Existing track → UPDATE
        if best_id is not None:
            track_id = best_id

        # No match → CREATE
        else:
            track_id = next_id
            next_id += 1

        # Update position
        updated_tracks[track_id] = centroid

        # Draw
        cv2.circle(
            frame,
            centroid,
            5,
            (255, 255, 255),
            -1
        )

        cv2.putText(
            frame,
            f"Person {track_id}",
            (centroid[0] + 10, centroid[1]),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    # IMPORTANT:
    # Current positions become previous positions
    tracks = updated_tracks

    cv2.imshow("Track Update", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [15]:
import cv2
from ultralytics import YOLO
import math

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")

next_id = 1
tracks = {}

MAX_DISTANCE = 80
MAX_MISSED = 10

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, classes=[0], verbose=False)

    current_centroids = []

    # -----------------------------
    # 1. Get current detections
    # -----------------------------
    for box in results[0].boxes.xyxy:

        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        current_centroids.append((cx, cy))

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

    # -----------------------------
    # 2. Track matching
    # -----------------------------
    updated_tracks = {}

    matched_ids = set()

    for centroid in current_centroids:

        best_id = None
        best_distance = MAX_DISTANCE

        for track_id, track in tracks.items():

            old_centroid = track["centroid"]

            distance = math.dist(
                centroid,
                old_centroid
            )

            if distance < best_distance:
                best_distance = distance
                best_id = track_id

        # -----------------------------
        # Existing track
        # -----------------------------
        if best_id is not None:

            track_id = best_id

            updated_tracks[track_id] = {
                "centroid": centroid,
                "missed": 0
            }

            matched_ids.add(track_id)

        # -----------------------------
        # New track
        # -----------------------------
        else:

            track_id = next_id
            next_id += 1

            updated_tracks[track_id] = {
                "centroid": centroid,
                "missed": 0
            }

        # Draw ID
        cv2.circle(
            frame,
            centroid,
            5,
            (255, 255, 255),
            -1
        )

        cv2.putText(
            frame,
            f"Person {track_id}",
            (centroid[0] + 10, centroid[1]),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    # -----------------------------
    # 3. Handle disappeared tracks
    # -----------------------------
    for track_id, track in tracks.items():

        if track_id not in matched_ids:

            track["missed"] += 1

            if track["missed"] <= MAX_MISSED:

                updated_tracks[track_id] = track

    # -----------------------------
    # 4. Save current tracks
    # -----------------------------
    tracks = updated_tracks

    cv2.imshow("Track Disappearance", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [17]:
import cv2
import math
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")

next_id = 1
tracks = {}

MAX_DISTANCE = 80
MAX_MISSED = 10

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(
        frame,
        classes=[0],
        verbose=False
    )

    # -----------------------------
    # 1. Get current person centroids
    # -----------------------------
    detections = []

    for box in results[0].boxes.xyxy:

        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        detections.append({
            "centroid": (cx, cy),
            "box": (x1, y1, x2, y2)
        })

    updated_tracks = {}
    used_track_ids = set()
    matched_detection_ids = set()

    # -----------------------------
    # 2. Match each detection
    # -----------------------------
    for detection_index, detection in enumerate(detections):

        centroid = detection["centroid"]

        best_id = None
        best_distance = MAX_DISTANCE

        for track_id, track in tracks.items():

            # Don't use the same track twice
            if track_id in used_track_ids:
                continue

            distance = math.dist(
                centroid,
                track["centroid"]
            )

            if distance < best_distance:
                best_distance = distance
                best_id = track_id

        # -----------------------------
        # Existing person
        # -----------------------------
        if best_id is not None:

            track_id = best_id

            updated_tracks[track_id] = {
                "centroid": centroid,
                "missed": 0
            }

            used_track_ids.add(track_id)
            matched_detection_ids.add(detection_index)

        # -----------------------------
        # New person
        # -----------------------------
        else:

            track_id = next_id
            next_id += 1

            updated_tracks[track_id] = {
                "centroid": centroid,
                "missed": 0
            }

            used_track_ids.add(track_id)
            matched_detection_ids.add(detection_index)

    # -----------------------------
    # 3. Handle disappeared people
    # -----------------------------
    for track_id, track in tracks.items():

        if track_id not in used_track_ids:

            track["missed"] += 1

            if track["missed"] <= MAX_MISSED:
                updated_tracks[track_id] = track

    tracks = updated_tracks

    # -----------------------------
    # 4. Draw everything
    # -----------------------------
    for detection in detections:

        x1, y1, x2, y2 = detection["box"]
        centroid = detection["centroid"]

        # Find ID belonging to this centroid
        person_id = None

        for track_id, track in tracks.items():

            if track["centroid"] == centroid:
                person_id = track_id
                break

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        cv2.circle(
            frame,
            centroid,
            5,
            (255, 255, 255),
            -1
        )

        if person_id is not None:

            cv2.putText(
                frame,
                f"Person {person_id}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

    cv2.imshow("Multi Object Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [21]:
import cv2
import numpy as np
from ultralytics import YOLO

# --------------------------------
# 1. Load YOLO
# --------------------------------
model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")


# --------------------------------
# 2. Create Kalman Filter
# --------------------------------
kalman = cv2.KalmanFilter(4, 2)

# State = [x, y, vx, vy]
kalman.transitionMatrix = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], dtype=np.float32)

# Measurement = [x, y]
kalman.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], dtype=np.float32)

# Noise settings
kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5

# Initial state
kalman.statePre = np.array([
    [0],
    [0],
    [0],
    [0]
], dtype=np.float32)

initialized = False


# --------------------------------
# 3. Process video
# --------------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # --------------------------------
    # YOLO: person only
    # --------------------------------
    results = model(
        frame,
        classes=[0],
        verbose=False
    )

    person_detected = False

    # Take first detected person
    if len(results[0].boxes.xyxy) > 0:

        box = results[0].boxes.xyxy[0]

        x1, y1, x2, y2 = map(int, box)

        # Calculate centroid
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        person_detected = True

        # --------------------------------
        # Initialize Kalman filter
        # --------------------------------
        if not initialized:

            kalman.statePre = np.array([
                [cx],
                [cy],
                [0],
                [0]
            ], dtype=np.float32)

            initialized = True

        # --------------------------------
        # Correct using YOLO measurement
        # --------------------------------
        measurement = np.array([
            [cx],
            [cy]
        ], dtype=np.float32)

        kalman.correct(measurement)

        # Draw YOLO detection
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        cv2.circle(
            frame,
            (cx, cy),
            5,
            (255, 255, 255),
            -1
        )

    # --------------------------------
    # Predict next position
    # --------------------------------
    if initialized:

        prediction = kalman.predict()

        predicted_x = int(prediction[0, 0])
        predicted_y = int(prediction[1, 0])

        # Draw prediction
        cv2.circle(
            frame,
            (predicted_x, predicted_y),
            8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            "Kalman Prediction",
            (predicted_x + 10, predicted_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    cv2.imshow(
        "YOLO + Kalman Filter",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

In [2]:
import cv2
import numpy as np
import math
from ultralytics import YOLO


# ---------------------------------------
# YOLO
# ---------------------------------------
model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")


# ---------------------------------------
# Parameters
# ---------------------------------------
MAX_DISTANCE = 100
MAX_MISSED = 10

next_id = 1
tracks = {}


# ---------------------------------------
# Create Kalman Filter
# ---------------------------------------
def create_kalman(cx, cy):

    kalman = cv2.KalmanFilter(4, 2)

    # State = [x, y, vx, vy]
    kalman.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    # Measurement = [x, y]
    kalman.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kalman.processNoiseCov = (
        np.eye(4, dtype=np.float32) * 0.03
    )

    kalman.measurementNoiseCov = (
        np.eye(2, dtype=np.float32) * 0.5
    )

    kalman.statePre = np.array([
        [cx],
        [cy],
        [0],
        [0]
    ], dtype=np.float32)

    return kalman


# ---------------------------------------
# Main loop
# ---------------------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break


    # -----------------------------------
    # YOLO person detection
    # -----------------------------------
    results = model(
        frame,
        classes=[0],
        verbose=False
    )


    detections = []


    # -----------------------------------
    # Extract centroids
    # -----------------------------------
    for box in results[0].boxes.xyxy:

        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        detections.append({
            "centroid": (cx, cy),
            "box": (x1, y1, x2, y2)
        })


    # -----------------------------------
    # Predict existing tracks
    # -----------------------------------
    predictions = {}

    for track_id, track in tracks.items():

        prediction = track["kalman"].predict()

        px = float(prediction[0, 0])
        py = float(prediction[1, 0])

        predictions[track_id] = (px, py)


    # -----------------------------------
    # Data Association
    # -----------------------------------
    updated_tracks = {}

    used_tracks = set()


    for detection in detections:

        centroid = detection["centroid"]

        best_id = None
        best_distance = MAX_DISTANCE


        for track_id, predicted_position in predictions.items():

            if track_id in used_tracks:
                continue

            distance = math.dist(
                centroid,
                predicted_position
            )

            if distance < best_distance:

                best_distance = distance
                best_id = track_id


        # --------------------------------
        # Existing track
        # --------------------------------
        if best_id is not None:

            track_id = best_id

            track = tracks[track_id]

            measurement = np.array([
                [centroid[0]],
                [centroid[1]]
            ], dtype=np.float32)

            track["kalman"].correct(measurement)

            track["centroid"] = centroid
            track["missed"] = 0

            updated_tracks[track_id] = track

            used_tracks.add(track_id)


        # --------------------------------
        # New track
        # --------------------------------
        else:

            track_id = next_id
            next_id += 1

            kalman = create_kalman(
                centroid[0],
                centroid[1]
            )

            updated_tracks[track_id] = {
                "kalman": kalman,
                "centroid": centroid,
                "missed": 0
            }

            used_tracks.add(track_id)


    # -----------------------------------
    # Handle disappeared tracks
    # -----------------------------------
    for track_id, track in tracks.items():

        if track_id not in used_tracks:

            track["missed"] += 1

            if track["missed"] <= MAX_MISSED:

                updated_tracks[track_id] = track


    tracks = updated_tracks


    # -----------------------------------
    # Draw detections + IDs
    # -----------------------------------
    for track_id, track in tracks.items():

        cx, cy = track["centroid"]

        # Draw centroid
        cv2.circle(
            frame,
            (cx, cy),
            5,
            (255, 255, 255),
            -1
        )

        # Draw ID
        cv2.putText(
            frame,
            f"Person {track_id}",
            (cx + 10, cy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )


    # Draw YOLO boxes
    for detection in detections:

        x1, y1, x2, y2 = detection["box"]

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )


    cv2.imshow(
        "YOLO + Multi Person Tracking",
        frame
    )


    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

In [7]:
import cv2
import numpy as np
import math
from ultralytics import YOLO


model = YOLO("yolo11n.pt")

cap = cv2.VideoCapture("assets/people-walking.mp4")


MAX_DISTANCE = 100
MAX_MISSED = 10

next_id = 1
tracks = {}


def create_kalman(cx, cy):

    kalman = cv2.KalmanFilter(4, 2)

    kalman.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], dtype=np.float32)

    kalman.measurementMatrix = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ], dtype=np.float32)

    kalman.processNoiseCov = (
        np.eye(4, dtype=np.float32) * 0.03
    )

    kalman.measurementNoiseCov = (
        np.eye(2, dtype=np.float32) * 0.5
    )

    kalman.statePre = np.array([
        [cx],
        [cy],
        [0],
        [0]
    ], dtype=np.float32)

    return kalman


while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(
        frame,
        classes=[0],
        verbose=False
    )

    detections = []

    # -----------------------------
    # Get detections
    # -----------------------------

    for box in results[0].boxes.xyxy:

        x1, y1, x2, y2 = map(int, box)

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        detections.append({
            "centroid": (cx, cy),
            "box": (x1, y1, x2, y2)
        })


    # -----------------------------
    # Predict existing tracks
    # -----------------------------

    predictions = {}

    for track_id, track in tracks.items():

        prediction = track["kalman"].predict()

        px = float(prediction[0, 0])
        py = float(prediction[1, 0])

        predictions[track_id] = (px, py)


    updated_tracks = {}
    used_tracks = set()


    # -----------------------------
    # Data Association
    # -----------------------------

    for detection in detections:

        centroid = detection["centroid"]

        best_id = None
        best_distance = MAX_DISTANCE

        for track_id, predicted_position in predictions.items():

            if track_id in used_tracks:
                continue

            distance = math.dist(
                centroid,
                predicted_position
            )

            if distance < best_distance:

                best_distance = distance
                best_id = track_id


        # -------------------------
        # Existing track
        # -------------------------

        if best_id is not None:

            track_id = best_id
            track = tracks[track_id]

            measurement = np.array([
                [centroid[0]],
                [centroid[1]]
            ], dtype=np.float32)

            track["kalman"].correct(measurement)

            track["centroid"] = centroid
            track["box"] = detection["box"]
            track["missed"] = 0
            track["state"] = "ACTIVE"

            updated_tracks[track_id] = track

            used_tracks.add(track_id)


        # -------------------------
        # New track
        # -------------------------

        else:

            track_id = next_id
            next_id += 1

            kalman = create_kalman(
                centroid[0],
                centroid[1]
            )

            updated_tracks[track_id] = {
                "kalman": kalman,
                "centroid": centroid,
                "box": detection["box"],
                "missed": 0,
                "state": "NEW"
            }

            used_tracks.add(track_id)


    # -----------------------------
    # Handle lost tracks
    # -----------------------------

    for track_id, track in tracks.items():

        if track_id not in used_tracks:

            track["missed"] += 1

            if track["missed"] <= MAX_MISSED:

                track["state"] = "LOST"

                updated_tracks[track_id] = track


    # -----------------------------
    # Delete old tracks
    # -----------------------------

    tracks = updated_tracks


    # -----------------------------
    # Visualization
    # -----------------------------

    for track_id, track in tracks.items():

        x1, y1, x2, y2 = track["box"]
        cx, cy = track["centroid"]

        # Bounding box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        # Centroid
        cv2.circle(
            frame,
            (cx, cy),
            5,
            (255, 255, 255),
            -1
        )

        # ID + state
        label = f"Person {track_id} | {track['state']}"

        cv2.putText(
            frame,
            label,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )


    cv2.imshow(
        "Track Management",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

In [9]:
import cv2
from ultralytics import YOLO

# --------------------------------
# 1. Load YOLO model
# --------------------------------
model = YOLO("yolo11n.pt")

# --------------------------------
# 2. Open video
# --------------------------------
cap = cv2.VideoCapture("assets/people-walking.mp4")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # --------------------------------
    # 3. YOLO + ByteTrack
    # --------------------------------
    results = model.track(
        frame,
        persist=True,
        classes=[0],
        tracker="bytetrack.yaml",
        verbose=False
    )

    result = results[0]

    # --------------------------------
    # 4. Draw tracked objects
    # --------------------------------
    if result.boxes is not None:

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id

        if ids is not None:

            ids = ids.cpu().numpy().astype(int)

            for box, track_id in zip(boxes, ids):

                x1, y1, x2, y2 = map(int, box)

                # Bounding box
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (255, 255, 255),
                    2
                )

                # Clean label position
                label = f"Person {track_id}"

                cv2.putText(
                    frame,
                    label,
                    (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA
                )

    # --------------------------------
    # 5. Display
    # --------------------------------
    cv2.imshow(
        "Professional Person Tracking",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

                    OBJECT TRACKING
                          │
             ┌────────────┴────────────┐
             ↓                         ↓
        YOLO Detection            Tracking
             │                         │
        Person only              Track IDs
             │                         │
          Boxes                 Data Association
             │                         │
        Centroids               Kalman / Motion
             │                         │
             └────────────┬────────────┘
                          ↓
                   Track Management
                          ↓
                Person 1 / Person 2
                Person 3 / Person 4
                          ↓
                     Visualization

In [10]:
import cv2
from ultralytics import YOLO


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_PATH = "yolo11n.pt"
INPUT_VIDEO = "assets/people-walking.mp4"
OUTPUT_VIDEO = "assets/people-tracking-final.mp4"

CONFIDENCE = 0.4
IMG_SIZE = 640


# ============================================================
# 2. LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)


# ============================================================
# 3. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Could not open input video")


# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))


# ============================================================
# 4. VIDEO WRITER
# ============================================================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    raise RuntimeError("Could not create output video")


# ============================================================
# 5. DISPLAY ID MANAGEMENT
# ============================================================

raw_to_display_id = {}
next_display_id = 1


# ============================================================
# 6. PROCESS VIDEO
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break


    # --------------------------------------------------------
    # YOLO + ByteTrack
    # --------------------------------------------------------

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[0],          # person only
        conf=CONFIDENCE,
        imgsz=IMG_SIZE,
        verbose=False
    )

    result = results[0]


    # --------------------------------------------------------
    # Get tracked boxes
    # --------------------------------------------------------

    if result.boxes is not None and result.boxes.is_track:

        boxes = result.boxes.xyxy.cpu().numpy()
        raw_ids = result.boxes.id.cpu().numpy().astype(int)


        # ----------------------------------------------------
        # Draw each tracked person
        # ----------------------------------------------------

        for box, raw_id in zip(boxes, raw_ids):

            x1, y1, x2, y2 = map(int, box)


            # -----------------------------------------------
            # Convert ByteTrack ID → clean display ID
            # -----------------------------------------------

            if raw_id not in raw_to_display_id:

                raw_to_display_id[raw_id] = next_display_id
                next_display_id += 1

            display_id = raw_to_display_id[raw_id]


            # -----------------------------------------------
            # Bounding box
            # -----------------------------------------------

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (255, 255, 255),
                2
            )


            # -----------------------------------------------
            # Label
            # -----------------------------------------------

            label = f"Person {display_id}"


            # Label background
            (text_w, text_h), _ = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                2
            )

            label_y = max(y1 - 8, text_h + 5)


            cv2.rectangle(
                frame,
                (x1, label_y - text_h - 5),
                (x1 + text_w + 8, label_y + 3),
                (0, 0, 0),
                -1
            )


            # Label text
            cv2.putText(
                frame,
                label,
                (x1 + 4, label_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )


    # --------------------------------------------------------
    # Write frame to output video
    # --------------------------------------------------------

    writer.write(frame)


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    cv2.imshow(
        "Professional Person Tracking",
        frame
    )


    # Press Q to stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ============================================================
# 7. CLEANUP
# ============================================================

cap.release()
writer.release()
cv2.destroyAllWindows()

print("Tracking finished.")
print(f"Saved video: {OUTPUT_VIDEO}")

Tracking finished.
Saved video: assets/people-tracking-final.mp4
